# LAB 3 — Análise Qualitativa: Impacto do Advanced RAG
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Medir e comparar a melhoria de qualidade das respostas após aplicar
query rewriting + reranking vs Naive RAG baseline da Aula 2.

**Entregáveis:**
- Comparação lado a lado: Naive RAG vs Advanced RAG para 5 queries
- Análise 5D de qualidade (Relevância, Cobertura, Precisão, Adequação, Fundamentação)
- Conclusão: qual melhoria teve maior impacto?

**Tempo estimado:** 30 minutos


## ⚙️ Passo 1 — Setup

In [ ]:
!pip install -q openai langchain-groq groq python-dotenv pandas numpy
import os, json, time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ── Provedores LLM: Groq primário + Ollama fallback ───────────────
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL   = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL    = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")

client = None
MODEL_NAME = None
LLM_PROVIDER = None

if GROQ_API_KEY:
    try:
        c = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        client, MODEL_NAME, LLM_PROVIDER = c, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM via Groq | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); usando Ollama…")

if client is None:
    try:
        c = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        c.chat.completions.create(model=OLLAMA_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        client, MODEL_NAME, LLM_PROVIDER = c, OLLAMA_LLM_MODEL, "ollama"
        print(f"✅ LLM via Ollama (fallback) | {MODEL_NAME}")
    except Exception as e:
        print(f"❌ Nenhum LLM disponível: {e}")


def llm(prompt, max_tokens=800):
    if client is None:
        return "[LLM indisponível]"
    try:
        r = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=0.1
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"[LLM indisponível: {e}]"

print(f"✅ Setup concluído (provider={LLM_PROVIDER})")

## 📋 Passo 2 — Carregar Resultados dos Labs Anteriores

In [ ]:
# Carregar dataset com expected answers
try:
    with open("../datasets/corpus_juridico_aula3.json", encoding="utf-8") as f:
        dataset = json.load(f)
    queries = dataset["queries_baseline"][:5]
    docs    = {d["id"]: d for d in dataset["documentos"]}
    print(f"✅ Dataset carregado: {len(queries)} queries, {len(docs)} documentos")
except Exception as e:
    print(f"⚠️  Erro ao carregar dataset: {e}")
    queries = [
        {"id": "q_001", "query_original": "Podem prender alguém sem mandado?",
         "resposta_esperada": "Sim, em caso de flagrante delito (art. 302 CPP). Audiência de custódia obrigatória em 24h.",
         "documentos_relevantes": ["doc_004", "doc_008"]},
        {"id": "q_002", "query_original": "Policial pode ler mensagens do celular do suspeito?",
         "resposta_esperada": "Não. HC 598.886/SC (STJ): exige autorização judicial.",
         "documentos_relevantes": ["doc_007"]},
        {"id": "q_003", "query_original": "O suspeito é obrigado a falar na delegacia?",
         "resposta_esperada": "Não. Direito ao silêncio (art. 5º LXIII CF).",
         "documentos_relevantes": ["doc_002"]},
        {"id": "q_004", "query_original": "O advogado pode ver o inquérito policial?",
         "resposta_esperada": "Sim. Súmula Vinculante 14 STF.",
         "documentos_relevantes": ["doc_005"]},
        {"id": "q_005", "query_original": "Como provar que o réu tinha intenção de matar?",
         "resposta_esperada": "Por elementos objetivos: laudo pericial, trajetória balística, dolo.",
         "documentos_relevantes": ["doc_006", "doc_010"]},
    ]
    docs = {}
    print("⚠️  Usando queries hardcoded")

## 🤖 Passo 3 — Simular Respostas (Naive RAG vs Advanced RAG)

In [ ]:
def gerar_contexto_naive(query, docs_dict, max_docs=3):
    """Simula contexto Naive RAG: primeiros 3 docs sem filtragem."""
    all_docs = list(docs_dict.values())[:max_docs]
    return "\n\n".join([f"[{d.get('numero', d['id'])}] {d['texto'][:300]}..." for d in all_docs])

def gerar_contexto_advanced(query, query_obj, docs_dict, max_docs=3):
    """Simula contexto Advanced RAG: documentos mais relevantes da query."""
    relevantes = query_obj.get("documentos_relevantes", [])
    contexto_docs = [docs_dict[did] for did in relevantes if did in docs_dict][:max_docs]
    if not contexto_docs:
        return gerar_contexto_naive(query, docs_dict, max_docs)
    return "\n\n".join([f"[{d.get('numero', d['id'])}] {d['texto'][:400]}..." for d in contexto_docs])

def gerar_resposta(query, contexto, modo):
    prompt = f"""Você é um assistente jurídico especializado em direito penal brasileiro.
Responda à pergunta baseando-se EXCLUSIVAMENTE no contexto fornecido.
Seja objetivo e cite a fonte quando possível.

CONTEXTO:
{contexto}

PERGUNTA: {query}

RESPOSTA:"""
    return llm(prompt, max_tokens=300)

print("⏳ Gerando respostas comparativas...")
print("   (Isso pode levar alguns minutos)")

comparacoes = []

for q in queries:
    qid = q["id"]
    query = q["query_original"]
    esperada = q.get("resposta_esperada", "N/A")
    
    print(f"  Processando {qid}...")
    
    ctx_naive    = gerar_contexto_naive(query, docs)
    ctx_advanced = gerar_contexto_advanced(query, q, docs)
    
    resp_naive    = gerar_resposta(query, ctx_naive, "naive")
    resp_advanced = gerar_resposta(query, ctx_advanced, "advanced")
    
    comparacoes.append({
        "qid": qid,
        "query": query,
        "esperada": esperada,
        "naive_rag": resp_naive,
        "advanced_rag": resp_advanced
    })

print("✅ Respostas geradas!")

## 📊 Passo 4 — Avaliação 5D de Qualidade

In [ ]:
def avaliar_5d(query, resposta_esperada, resposta_gerada):
    """
    Avaliação automatizada em 5 dimensões (0-2 pontos cada):
    1. Relevância: responde à pergunta?
    2. Cobertura: aborda os aspectos principais?
    3. Precisão: informações corretas sem alucinações?
    4. Adequação jurídica: usa terminologia e fontes corretas?
    5. Fundamentação: cita base legal ou jurisprudência?
    """
    prompt = f"""Avalie a resposta gerada comparando com a esperada.
Atribua pontuação 0, 1 ou 2 para cada dimensão:
0 = ausente/incorreto | 1 = parcial | 2 = completo/correto

PERGUNTA: {query}
RESPOSTA ESPERADA: {resposta_esperada}
RESPOSTA GERADA: {resposta_gerada}

Avalie e retorne EXATAMENTE neste formato JSON:
{{"relevancia": X, "cobertura": X, "precisao": X, "adequacao_juridica": X, "fundamentacao": X, "comentario": "..."}}"""
    
    result = llm(prompt, max_tokens=200)
    try:
        import re
        json_match = re.search(r'\{.*\}', result, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass
    # Fallback
    return {"relevancia": 1, "cobertura": 1, "precisao": 1, "adequacao_juridica": 1, "fundamentacao": 1, "comentario": "Avaliação manual necessária"}

print("⏳ Avaliando respostas em 5 dimensões...")

resultados_avaliacao = []

for c in comparacoes:
    av_naive    = avaliar_5d(c["query"], c["esperada"], c["naive_rag"])
    av_advanced = avaliar_5d(c["query"], c["esperada"], c["advanced_rag"])
    
    score_naive    = sum([av_naive.get(k, 0) for k in ["relevancia","cobertura","precisao","adequacao_juridica","fundamentacao"]])
    score_advanced = sum([av_advanced.get(k, 0) for k in ["relevancia","cobertura","precisao","adequacao_juridica","fundamentacao"]])
    
    melhoria = score_advanced - score_naive
    
    resultados_avaliacao.append({
        **c,
        "score_naive": score_naive,
        "score_advanced": score_advanced,
        "melhoria": melhoria,
        "av_naive": av_naive,
        "av_advanced": av_advanced
    })
    
    print(f"[{c['qid']}] Naive: {score_naive}/10 | Advanced: {score_advanced}/10 | Δ: +{melhoria}")

print("\n✅ Avaliação concluída!")

## 📈 Passo 5 — Visualização dos Resultados

In [ ]:
print("📊 RESUMO DA COMPARAÇÃO NAIVE RAG vs ADVANCED RAG")
print("=" * 65)
print()
print(f"{'Query':8} {'Naive':8} {'Advanced':10} {'Melhoria':10} {'Status'}")
print("-" * 55)

total_naive = 0
total_advanced = 0

for r in resultados_avaliacao:
    melhoria_str = f"+{r['melhoria']}" if r['melhoria'] >= 0 else str(r['melhoria'])
    status = "✅ Melhorou" if r['melhoria'] > 0 else ("✅ Igual" if r['melhoria'] == 0 else "⚠️  Piorou")
    print(f"{r['qid']:8} {r['score_naive']:5}/10  {r['score_advanced']:5}/10   {melhoria_str:8}   {status}")
    total_naive    += r['score_naive']
    total_advanced += r['score_advanced']

n = len(resultados_avaliacao)
print("-" * 55)
print(f"{'MÉDIA':8} {total_naive/n:.1f}/10   {total_advanced/n:.1f}/10   {(total_advanced-total_naive)/n:+.1f}")
print()

melhoria_media = (total_advanced - total_naive) / n
print(f"📌 Melhoria média: {melhoria_media:+.1f} pontos em escala 0-10")
print(f"   Ganho percentual: +{(total_advanced - total_naive) / max(total_naive, 1) * 100:.1f}%")

## ✅ Passo 6 — Conclusão

In [ ]:
print("📝 ANÁLISE DE IMPACTO POR MELHORIA")
print("=" * 55)
print()
print("Qual técnica teve maior impacto nas respostas?")
print()
print("Para avaliar, analise as dimensões individuais:")
print()
for r in resultados_avaliacao:
    print(f"[{r['qid']}]")
    for dim in ["relevancia", "cobertura", "precisao", "adequacao_juridica", "fundamentacao"]:
        delta = r["av_advanced"].get(dim, 0) - r["av_naive"].get(dim, 0)
        bar = "+" * max(0, delta) + "-" * max(0, -delta)
        print(f"  {dim:20s}: {r['av_naive'].get(dim,0)} → {r['av_advanced'].get(dim,0)} [{bar}]")
    print()

print()
print("✅ LAB 3 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] Respostas geradas para Naive RAG e Advanced RAG")
print("   [ ] Avaliação 5D realizada para todas as queries")
print("   [ ] Tabela comparativa com scores preenchida")
print("   [ ] Conclusão escrita sobre qual melhoria foi mais impactante")